# 04. Speech Emotion Recognition

Extract 162-dimensional speech features using the exact backend preprocessing logic, train an SVM, and save the artifacts used by the API.


In [1]:
from pathlib import Path
import sys

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "NeuroSense" / "webdev" / "backend").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root for this notebook.")

PROJECT_ROOT = _find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "NeuroSense" / "notebooks"
BACKEND_DIR = PROJECT_ROOT / "NeuroSense" / "webdev" / "backend"

for path in (NOTEBOOKS_DIR, BACKEND_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from notebook_support import bootstrap_notebook

ctx = bootstrap_notebook(PROJECT_ROOT)
DATASETS_DIR = ctx["datasets_dir"]
ARTIFACTS_DIR = ctx["artifacts_dir"]
CACHE_DIR = ctx["cache_dir"]
RANDOM_STATE = ctx["random_state"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Datasets directory: {DATASETS_DIR}")
print(f"Artifacts directory: {ARTIFACTS_DIR}")


Project root: /Users/devashishsingh/Desktop/human emotion recognition system
Datasets directory: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/datasets
Artifacts directory: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/artifacts


In [2]:
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

from notebook_support import collect_audio_paths, extract_feature_dataset, label_counts, sample_records_by_label
from utils.emotion_utils import map_emotion_to_sentiment
from utils.preprocessors import preprocess_speech

def normalize_speech_label(folder_name):
    normalized = folder_name.lower().replace("-", "_").replace(" ", "_")
    if "pleasant" in normalized or normalized.endswith("_ps") or normalized == "ps":
        return "ps"
    for emotion in ("angry", "disgust", "fear", "happy", "neutral", "sad"):
        if emotion in normalized:
            return emotion
    return None

speech_root = DATASETS_DIR / "speech"
if not speech_root.exists():
    raise FileNotFoundError(f"Missing speech dataset directory: {speech_root}")

MAX_FILES_PER_EMOTION = 150

records = collect_audio_paths(speech_root, normalize_speech_label)
records = sample_records_by_label(records, per_label=MAX_FILES_PER_EMOTION, seed=RANDOM_STATE)
print("Speech label counts:", label_counts(label for _, label in records))
print("Speech sentiment counts:", label_counts(map_emotion_to_sentiment(label) for _, label in records))

X, y_raw = extract_feature_dataset(
    records,
    preprocess_speech,
    cache_path=CACHE_DIR / "speech_features.npz",
    progress_interval=100,
)

print("Speech feature matrix:", X.shape)


Speech label counts: {'angry': 150, 'disgust': 150, 'fear': 150, 'happy': 150, 'neutral': 150, 'ps': 150, 'sad': 150}
Speech sentiment counts: {'NEGATIVE': 600, 'NEUTRAL': 150, 'POSITIVE': 300}
Speech feature matrix: (1050, 162)


In [3]:
le = LabelEncoder()
y = le.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

model = SVC(kernel="rbf", C=5.0, probability=True, random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(
    make_pipeline(StandardScaler(), SVC(kernel="rbf", C=5.0, probability=True, random_state=RANDOM_STATE)),
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)

print(f"Speech test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Speech CV mean: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(classification_report(y_test, y_pred, target_names=le.classes_))


Speech test accuracy: 1.0000
Speech CV mean: 0.9988 +/- 0.0017
              precision    recall  f1-score   support

       angry       1.00      1.00      1.00        30
     disgust       1.00      1.00      1.00        30
        fear       1.00      1.00      1.00        30
       happy       1.00      1.00      1.00        30
     neutral       1.00      1.00      1.00        30
          ps       1.00      1.00      1.00        30
         sad       1.00      1.00      1.00        30

    accuracy                           1.00       210
   macro avg       1.00      1.00      1.00       210
weighted avg       1.00      1.00      1.00       210



In [4]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="magma", xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Speech confusion matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

feature_summary = pd.DataFrame(X[:5]).T.describe().T[["mean", "std"]].head(10)
feature_summary


/var/folders/y1/kwl357md29bd8ggvss23h_yc0000gn/T/ipykernel_59790/3216124583.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,mean,std
0,34.985157,394.593842
1,21.825207,224.328186
2,32.294472,359.526917
3,29.681366,323.979828
4,31.421999,352.016937


In [5]:
artifact_dir = ARTIFACTS_DIR / "speech"
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, artifact_dir / "speech_model.pkl")
joblib.dump(scaler, artifact_dir / "speech_scaler.pkl")
joblib.dump(le, artifact_dir / "speech_label_encoder.pkl")

print("Saved speech artifacts to:", artifact_dir)


Saved speech artifacts to: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/artifacts/speech
